<a href="https://colab.research.google.com/github/jungeun1228/2026_08_Tahoe-100M_scRNA-seq-analysis/blob/main/01_Tahoe_AnnData_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q -U anndata pyarrow scipy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 730.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy import sparse
import anndata as ad

PROJECT = Path("/content/drive/MyDrive/Tahoe_project")

RAW = PROJECT / "raw_parts"
OUT = PROJECT / "anndata"

OUT.mkdir(parents=True, exist_ok=True)

files = sorted(RAW.glob("batch_*.parquet"))

print("Parquet files:", len(files))
print("Gene metadata:", (PROJECT / "gene_metadata.parquet").exists())
print("Sample metadata:", (PROJECT / "selected_samples.csv").exists())

assert len(files) > 0, "No Parquet files found"

Parquet files: 340
Gene metadata: True
Sample metadata: True


In [ ]:
# Read Gene metadata
gene_meta = pd.read_parquet(
    PROJECT / "gene_metadata.parquet"
)

print("Gene metadata shape:", gene_meta.shape)

print("\nColumns:")
print(gene_meta.columns.tolist())

display(gene_meta.head())

Gene metadata shape: (62710, 3)

Columns:
['gene_symbol', 'ensembl_id', 'token_id']


,gene_symbol,ensembl_id,token_id
0,TSPAN6,ENSG00000000003,3
1,TNMD,ENSG00000000005,4
2,DPM1,ENSG00000000419,5
3,SCYL3,ENSG00000000457,6
4,C1orf112,ENSG00000000460,7


In [ ]:
# Check a parquet file
pf = pq.ParquetFile(files[0])

print("Parquet columns:")
print(pf.schema_arrow.names)

batch = next(
    pf.iter_batches(batch_size=1)
)

cell = batch.to_pylist()[0]

print("\nCell metadata:")

for key, value in cell.items():

    if key not in ["genes", "expressions"]:
        print(key, ":", value)

print("\nFirst 10 gene tokens:")
print(cell["genes"][:10])

print("\nFirst 10 expression values:")
print(cell["expressions"][:10])

print("\nNumber of entries:")
print(len(cell["genes"]))
print(len(cell["expressions"]))

Parquet columns:
['cell_id', 'sample', 'drug', 'plate', 'cell_line_id', 'moa_fine', 'genes', 'expressions']

Cell metadata:
cell_id : 02_001_163-lib_1105
sample : smp_1784
drug : Chlorhexidine (diacetate)
plate : plate4
cell_line_id : CVCL_1098
moa_fine : unclear

First 10 gene tokens:
[1, 19, 22, 43, 70, 112, 128, 139, 146, 174]

First 10 expression values:
[-2.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0]

Number of entries:
684
684


In [ ]:
# Test count matrix conversion (for 500 cells)
# Gene metadata preparation

required = {"token_id", "ensembl_id", "gene_symbol"}

assert required.issubset(gene_meta.columns)
assert gene_meta["token_id"].is_unique
assert gene_meta["ensembl_id"].is_unique

gene_meta = (
    gene_meta
    .sort_values("token_id")
    .reset_index(drop=True)
)

token_to_col = {
    int(token): i
    for i, token in enumerate(gene_meta["token_id"])
}

n_genes = len(gene_meta)

# Sparse matrix components
data = []
indices = []
indptr = [0]

obs_records = []

N_TEST = 500

for file in files:

    pf = pq.ParquetFile(file)

    for batch in pf.iter_batches(batch_size=256):

        for cell in batch.to_pylist():

            genes = cell["genes"]
            counts = cell["expressions"]

            assert len(genes) == len(counts)

            # Remove Tahoe CLS marker
            genes = genes[1:]
            counts = counts[1:]

            unknown = set(genes) - token_to_col.keys()

            if unknown:
                raise ValueError(
                    f"Unknown gene tokens: {list(unknown)[:5]}"
                )

            cols = [
                token_to_col[int(g)]
                for g in genes
            ]

            values = np.asarray(
                counts,
                dtype=np.float32
            )

            assert np.isfinite(values).all()
            assert (values >= 0).all()

            indices.extend(cols)
            data.extend(values)

            indptr.append(len(data))

            obs_records.append({
                k: v
                for k, v in cell.items()
                if k not in ["genes", "expressions"]
            })

            if len(obs_records) >= N_TEST:
                break

        if len(obs_records) >= N_TEST:
            break

    if len(obs_records) >= N_TEST:
        break


# Construct CSR sparse matrix

X = sparse.csr_matrix(
    (
        np.asarray(data, dtype=np.float32),
        np.asarray(indices, dtype=np.int32),
        np.asarray(indptr, dtype=np.int64)
    ),
    shape=(len(obs_records), n_genes)
)

X.sum_duplicates()

# Construct AnnData

obs = pd.DataFrame(obs_records)

var = gene_meta[
    ["ensembl_id", "gene_symbol", "token_id"]
].copy()

var = var.set_index("ensembl_id")

adata_test = ad.AnnData(
    X=X,
    obs=obs,
    var=var
)

# Set unique cell IDs

assert adata_test.obs["cell_id"].is_unique

adata_test.obs_names = (
    adata_test.obs["cell_id"].astype(str)
)

print(adata_test)

AnnData object with n_obs × n_vars = 500 × 62710
    obs: 'cell_id', 'sample', 'drug', 'plate', 'cell_line_id', 'moa_fine'
    var: 'gene_symbol', 'token_id'
    layers: None (.X)


/usr/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
# Check if conversion was successful
print("Cells:", adata_test.n_obs)
print("Genes:", adata_test.n_vars)

print("Sparse matrix:", sparse.issparse(adata_test.X))

print("Matrix shape:", adata_test.X.shape)

print("Non-zero values:", adata_test.X.nnz)

print("Total counts:", adata_test.X.sum())

print("\nCell metadata:")
display(adata_test.obs.head())

print("\nGene metadata:")
display(adata_test.var.head())

Cells: 500
Genes: 62710
Sparse matrix: True
Matrix shape: (500, 62710)
Non-zero values: 572314
Total counts: 839597.0

Cell metadata:


,cell_id,sample,drug,plate,cell_line_id,moa_fine
cell_id,,,,,,
02_001_163-lib_1105,02_001_163-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_1098,unclear
02_006_070-lib_1105,02_006_070-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_1098,unclear
02_006_146-lib_1105,02_006_146-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0131,unclear
02_007_062-lib_1105,02_007_062-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0131,unclear
02_009_069-lib_1105,02_009_069-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0131,unclear



Gene metadata:


,gene_symbol,token_id
ensembl_id,,
ENSG00000000003,TSPAN6,3
ENSG00000000005,TNMD,4
ENSG00000000419,DPM1,5
ENSG00000000457,SCYL3,6
ENSG00000000460,C1orf112,7


In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [ ]:
# Copy parquet files to Colab local disk
import shutil
import os
from pathlib import Path
from tqdm.auto import tqdm
import pyarrow.parquet as pq

PROJECT = Path("/content/drive/MyDrive/Tahoe_project")

RAW = PROJECT / "raw_parts"
LOCAL = Path("/content/tahoe_raw")

LOCAL.mkdir(parents=True, exist_ok=True)

files = sorted(RAW.glob("batch_*.parquet"))

assert len(files) == 340, "Drive connection or file count problem"

print(f"Total files: {len(files)}")

copied = 0
skipped = 0

for f in tqdm(files):

    dest = LOCAL / f.name
    temp = LOCAL / (f.name + ".part")

    source_size = f.stat().st_size

    # Skip previously copied files only if readable
    if dest.exists():

        if dest.stat().st_size == source_size:

            try:
                pq.ParquetFile(dest).metadata
                skipped += 1
                continue

            except Exception:
                pass

    # Remove incomplete temporary copy
    if temp.exists():
        temp.unlink()

    # Copy into a temporary file
    shutil.copy2(f, temp)

    # Validate copy
    assert temp.stat().st_size == source_size

    pq.ParquetFile(temp).metadata

    # Replace destination only after successful copy
    os.replace(temp, dest)

    copied += 1

print("\n===== COPY COMPLETED =====")
print("Newly copied:", copied)
print("Previously completed:", skipped)
print("Total:", copied + skipped)

Total files: 340


  0%|          | 0/340 [00:00<?, ?it/s]


===== COPY COMPLETED =====
Newly copied: 337
Previously completed: 3
Total: 340


In [ ]:
# Prepare Gene & sample metadata
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad
import pyarrow.dataset as ds
import gc

gene_meta = pd.read_parquet(
    PROJECT / "gene_metadata.parquet"
)

samples = pd.read_csv(
    PROJECT / "selected_samples.csv"
)

# Gene metadata validation
assert gene_meta["token_id"].is_unique
assert gene_meta["ensembl_id"].is_unique
assert samples["sample"].is_unique

gene_meta = (
    gene_meta
    .sort_values("token_id")
    .reset_index(drop=True)
)

# Fast token -> column lookup
max_token = int(gene_meta["token_id"].max())

lookup = np.full(
    max_token + 1,
    -1,
    dtype=np.int32
)

lookup[
    gene_meta["token_id"].to_numpy(dtype=np.int64)
] = np.arange(len(gene_meta), dtype=np.int32)

n_genes = len(gene_meta)

var = (
    gene_meta
    .set_index("ensembl_id")[
        ["gene_symbol", "token_id"]
    ]
    .copy()
)

# Sample metadata
samples["is_control"] = (
    samples["is_control"]
    .astype(str)
    .str.lower()
    .eq("true")
)

sample_info = samples.set_index("sample")

CELL_LINES = {
    "CVCL_0320": "HT29",
    "CVCL_0480": "PANC1",
    "CVCL_1098": "HepG2_C3A",
    "CVCL_0131": "A172",
    "CVCL_0179": "BT474",
}

print("Genes:", n_genes)
print("Samples:", len(samples))
print("Cell lines:", len(CELL_LINES))

Genes: 62710
Samples: 85
Cell lines: 5


In [ ]:
# Create AnnData by cell line
def build_anndata(cell_id, cell_name):

    output = OUT / f"{cell_name}_raw.h5ad"

    # Already completed?
    if output.exists():
        try:
            saved = ad.read_h5ad(output, backed="r")
            shape = saved.shape
            saved.file.close()

            if shape[1] == n_genes:
                print(f"Skipping {cell_name}: {shape}")
                return

        except Exception:
            print("Existing file is invalid. Rebuilding.")

    print(f"\nProcessing {cell_name}...", flush=True)

    paths = sorted(LOCAL.glob("batch_*.parquet"))

    dataset = ds.dataset(
        [str(p) for p in paths],
        format="parquet"
    )

    scanner = dataset.scanner(
        filter=ds.field("cell_line_id") == cell_id,
        batch_size=512,
        columns=[
            "cell_id",
            "sample",
            "drug",
            "plate",
            "cell_line_id",
            "moa_fine",
            "genes",
            "expressions"
        ]
    )

    chunks = []
    obs_records = []

    total_cells = 0

    for batch in scanner.to_batches():
        # Skip empty batches after cell-line filtering
        if batch.num_rows == 0:
            continue

        rows = batch.to_pylist()

        data_parts = []
        index_parts = []
        indptr = [0]

        for cell in rows:

            tokens = np.asarray(
                cell["genes"],
                dtype=np.int64
            )

            values = np.asarray(
                cell["expressions"],
                dtype=np.float32
            )

            if len(tokens) != len(values) or len(tokens) == 0:
                raise ValueError("Invalid gene/count arrays")

            # Remove CLS marker
            tokens = tokens[1:]
            values = values[1:]

            # Validate tokens
            if len(tokens) and (
                tokens.min() < 0 or tokens.max() > max_token
            ):
                raise ValueError("Gene token out of range")

            cols = lookup[tokens]

            if np.any(cols < 0):
                raise ValueError("Unknown gene token")

            # Validate raw counts
            if not np.all(np.isfinite(values)):
                raise ValueError("Non-finite counts")

            if np.any(values < 0):
                raise ValueError("Negative counts")

            if not np.all(values == np.floor(values)):
                raise ValueError("Non-integer raw counts")

            data_parts.append(values)
            index_parts.append(cols)

            indptr.append(
                indptr[-1] + len(values)
            )

            obs_records.append({
                k: v
                for k, v in cell.items()
                if k not in ["genes", "expressions"]
            })

        # Construct sparse chunk
        X_chunk = sparse.csr_matrix(
            (
                np.concatenate(data_parts),
                np.concatenate(index_parts),
                np.asarray(indptr, dtype=np.int64)
            ),
            shape=(len(rows), n_genes)
        )

        X_chunk.sum_duplicates()

        chunks.append(X_chunk)

        total_cells += len(rows)

        if total_cells % 20000 < 512:
            print(
                f"{cell_name}: {total_cells:,} cells",
                flush=True
            )

    if not chunks:
        raise ValueError(f"No cells found for {cell_name}")

    # Combine sparse chunks
    X = sparse.vstack(
        chunks,
        format="csr"
    )

    obs = pd.DataFrame(obs_records)

    # Add concentration and control metadata
    obs = obs.join(
        sample_info[
            ["concentration", "is_control"]
        ],
        on="sample",
        validate="many_to_one"
    )

    assert obs["is_control"].notna().all()
    assert obs["cell_id"].is_unique
    assert (obs["cell_line_id"] == cell_id).all()

    # Validate sample-to-treatment mapping
    expected_drug = obs["sample"].map(
        sample_info["drug"]
    )

    assert (obs["drug"] == expected_drug).all()

    expected_plate = obs["sample"].map(
        sample_info["plate"]
    )

    assert (
        obs["plate"].astype(str)
        == expected_plate.astype(str)
    ).all()

    obs["cell_line"] = cell_name

    obs.index = pd.Index(
        obs["cell_id"].astype(str),
        name="cell_id"
    )

    # Create AnnData
    adata = ad.AnnData(
        X=X,
        obs=obs,
        var=var.copy()
    )

    # Store unnormalized raw counts in X.
    # No duplicate counts layer yet to save memory.

    print(
        f"Saving {cell_name}: {adata.shape}",
        flush=True
    )

    local_output = (
        Path("/content")
        / f"{cell_name}_raw.h5ad"
    )

    adata.write_h5ad(
        local_output,
        compression="lzf"
    )

    # Copy to Drive via a temporary filename.
    # An interrupted copy will not replace the completed file.
    temporary = OUT / f"{cell_name}_raw.partial.h5ad"
    shutil.copy2(local_output, temporary)
    temporary.replace(output)

    # Cleanup
    del adata, X, chunks, obs, obs_records
    gc.collect()

    print(f"Completed: {output}", flush=True)

In [ ]:
# Try first with HT29
build_anndata(
    "CVCL_0320",
    "HT29"
)


Processing HT29...
HT29: 31 cells
HT29: 128 cells
HT29: 215 cells
HT29: 306 cells
HT29: 416 cells
HT29: 451 cells
HT29: 20,014 cells
HT29: 20,103 cells
HT29: 20,193 cells
HT29: 20,276 cells
HT29: 20,357 cells
HT29: 20,401 cells
HT29: 20,490 cells
HT29: 40,098 cells
HT29: 40,116 cells
HT29: 40,214 cells
HT29: 40,315 cells
HT29: 40,415 cells
HT29: 60,030 cells
HT29: 60,127 cells
HT29: 60,227 cells
HT29: 60,326 cells
HT29: 60,413 cells
HT29: 80,014 cells
HT29: 80,114 cells
HT29: 80,226 cells
HT29: 80,269 cells
HT29: 80,354 cells
HT29: 80,450 cells
HT29: 100,014 cells
HT29: 100,102 cells
HT29: 100,135 cells
HT29: 100,227 cells
HT29: 100,315 cells
HT29: 100,413 cells
HT29: 120,063 cells
HT29: 120,177 cells
HT29: 120,277 cells
HT29: 120,387 cells
HT29: 120,398 cells
HT29: 120,493 cells
HT29: 140,074 cells
HT29: 140,132 cells
HT29: 140,225 cells
HT29: 140,301 cells
HT29: 140,403 cells
HT29: 140,480 cells
Saving HT29: (142046, 62710)
Completed: /content/drive/MyDrive/Tahoe_project/anndata/HT2

In [ ]:
# Check
adata = ad.read_h5ad(
    OUT / "HT29_raw.h5ad"
)

print(adata)

print("\nCells:", adata.n_obs)
print("Genes:", adata.n_vars)
print("Sparse:", sparse.issparse(adata.X))

print("\nMetadata:")
display(adata.obs.head())

print("\nGene metadata:")
display(adata.var.head())

AnnData object with n_obs × n_vars = 142046 × 62710
    obs: 'cell_id', 'sample', 'drug', 'plate', 'cell_line_id', 'moa_fine', 'concentration', 'is_control', 'cell_line'
    var: 'gene_symbol', 'token_id'
    layers: None (.X)

Cells: 142046
Genes: 62710
Sparse: True

Metadata:


,cell_id,sample,drug,plate,cell_line_id,moa_fine,concentration,is_control,cell_line
cell_id,,,,,,,,,
02_020_154-lib_1105,02_020_154-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0320,unclear,0.05,False,HT29
02_033_109-lib_1105,02_033_109-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0320,unclear,0.05,False,HT29
02_036_144-lib_1105,02_036_144-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0320,unclear,0.05,False,HT29
02_092_067-lib_1105,02_092_067-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0320,unclear,0.05,False,HT29
02_092_164-lib_1105,02_092_164-lib_1105,smp_1784,Chlorhexidine (diacetate),plate4,CVCL_0320,unclear,0.05,False,HT29



Gene metadata:


,gene_symbol,token_id
ensembl_id,,
ENSG00000000003,TSPAN6,3
ENSG00000000005,TNMD,4
ENSG00000000419,DPM1,5
ENSG00000000457,SCYL3,6
ENSG00000000460,C1orf112,7


In [ ]:
# Proceed with the rest 4 cell lines
for cell_id, cell_name in CELL_LINES.items():

    build_anndata(
        cell_id,
        cell_name
    )

Skipping HT29: (142046, 62710)

Processing PANC1...
PANC1: 37 cells
PANC1: 211 cells
PANC1: 361 cells
PANC1: 20,102 cells
PANC1: 20,262 cells
PANC1: 20,305 cells
PANC1: 20,464 cells
PANC1: 40,047 cells
PANC1: 40,067 cells
PANC1: 40,212 cells
PANC1: 40,348 cells
PANC1: 40,485 cells
PANC1: 60,202 cells
PANC1: 60,473 cells
PANC1: 80,207 cells
PANC1: 80,441 cells
PANC1: 80,494 cells
PANC1: 100,056 cells
PANC1: 100,287 cells
PANC1: 120,035 cells
PANC1: 120,284 cells
PANC1: 140,098 cells
PANC1: 140,327 cells
PANC1: 160,224 cells
PANC1: 160,325 cells
PANC1: 180,107 cells
PANC1: 180,315 cells
PANC1: 200,205 cells
PANC1: 200,421 cells
PANC1: 220,110 cells
PANC1: 220,346 cells
PANC1: 240,112 cells
PANC1: 240,229 cells
PANC1: 240,473 cells
PANC1: 260,049 cells
PANC1: 260,189 cells
PANC1: 260,320 cells
PANC1: 260,397 cells
PANC1: 280,034 cells
PANC1: 280,183 cells
PANC1: 280,325 cells
PANC1: 280,395 cells
Saving PANC1: (288084, 62710)
Completed: /content/drive/MyDrive/Tahoe_project/anndata/PANC1_r

In [ ]:
import anndata as ad

total_cells = 0

for cell_id, cell_name in CELL_LINES.items():

    path = OUT / f"{cell_name}_raw.h5ad"

    assert path.exists(), f"Missing file: {cell_name}"

    adata = ad.read_h5ad(path, backed="r")

    print(
        cell_name,
        adata.n_obs,
        "cells ×",
        adata.n_vars,
        "genes"
    )

    total_cells += adata.n_obs

    adata.file.close()

print("\nTotal cells:", f"{total_cells:,}")

assert total_cells == 742_008, (
    f"Cell count mismatch: {total_cells:,}"
)

print("Cell count validation PASSED!")

HT29 142046 cells × 62710 genes
PANC1 288084 cells × 62710 genes
HepG2_C3A 77915 cells × 62710 genes
A172 151159 cells × 62710 genes
BT474 82804 cells × 62710 genes

Total cells: 742,008
Cell count validation PASSED!


In [ ]:
# Total cells: 742,008
# Total genes: 62,710